# APUSH LEQ Grader SLM — v2 gated training and evaluation

This Colab notebook builds and trains only the audited **data-improvement v2** artifacts. It deliberately rejects the legacy root-level `train_chat_v2.jsonl`, requires completed synthetic-label review before training, and keeps litmus, challenge, external, and permission-gated College Board evaluation separate.

Current local status: 87 independently accepted realistic cases, 9 pending review records, and no release-ready v2 training artifact yet.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Attach a GPU runtime before continuing."

## 2. Clone an explicit repository revision

Set `GIT_REF` to the commit containing the v2 scripts and artifacts. A commit SHA is strongly preferred over `main` for reproducibility.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aryanjverma/slm.git"
REPO_DIR = Path("/content/slm")
GIT_REF = os.environ.get("SLM_GIT_REF", "main")  # Prefer a committed SHA.

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", commit)

required = [
    Path("scripts/build_v2_artifacts.py"),
    Path("scripts/review_synthetic_v2.py"),
    Path("scripts/run_v2_checkpoints.py"),
    Path("scripts/eval_hf_model.py"),
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f"The selected revision does not contain the v2 pipeline: {missing}"

## 3. Install dependencies

In [ ]:
!pip install -q -e ".[train]" sentencepiece tiktoken

## 4. Optional Hugging Face token

In [ ]:
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("No HF_TOKEN configured; downloads will be anonymous.")
except Exception as exc:
    print("No Colab HF_TOKEN available:", exc)

## 5. Configure the v2 run

The canonical training input is `artifacts/data/v2/train_chat_v2.jsonl`, produced by the audited builder. The similarly named root-level 800-row file is a legacy template oversample and is never selected here.

In [ ]:
import json, math, hashlib, sys
from collections import Counter

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = "apush_frq_grader_v2"
MODEL_DIR = Path("artifacts/models/apush-frq-grader-v2")
EVAL_DIR = Path("artifacts/eval/v2")
V2_DIR = Path("artifacts/data/v2")

UNREVIEWED = Path("artifacts/data/train_realistic_v2_unreviewed.jsonl")
REVIEW_LOG = Path("artifacts/reviews/synthetic_v2.jsonl")
REVIEWED = Path("artifacts/data/train_realistic_v2_reviewed.jsonl")
LEGACY_ADVERSARIAL = Path("artifacts/data/train_cases.jsonl")
LEGACY_TEMPLATE_V2 = Path("artifacts/data/train_chat_v2.jsonl")
TRAIN_CHAT = V2_DIR / "train_chat_v2.jsonl"
MANIFEST = V2_DIR / "dataset_manifest_v2.json"

LITMUS = Path("artifacts/data/eval_cases.jsonl")
CHALLENGE = V2_DIR / "eval_challenge_v2.jsonl"
GOLDEN = V2_DIR / "eval_cb_golden_v2.jsonl"
EXTERNAL = V2_DIR / "eval_external_v2.jsonl"
PERMISSION = Path("config/college_board_permission.json")

ADVERSARIAL_RATIO = 0.15
TARGET_CAP = 1200
CHECKPOINT_COUNTS = [200, 500, 1200]
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
RUN_BASE_EVAL = True
RUN_CHECKPOINT_SWEEP = False
FORCE_REBUILD = False

EVAL_DIR.mkdir(parents=True, exist_ok=True)

## 6. Inventory available data

In [ ]:
def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def row_count(path):
    return len(read_jsonl(path))

print(f"Legacy v1 chat rows:       {row_count(Path('artifacts/data/train_chat.jsonl'))}")
print(f"Legacy template-v2 rows:  {row_count(LEGACY_TEMPLATE_V2)} (not canonical v2)")
print(f"Accepted realistic rows:  {row_count(UNREVIEWED)} (not train-ready until review)")
print(f"Completed reviewed rows:  {row_count(REVIEWED)}")
print(f"Canonical v2 chat rows:   {row_count(TRAIN_CHAT)}")
print(f"Litmus eval rows:         {row_count(LITMUS)}")
print(f"Challenge eval rows:      {row_count(CHALLENGE)}")
print(f"Golden eval rows:         {row_count(GOLDEN)}")
print(f"External eval rows:       {row_count(EXTERNAL)}")

if LEGACY_TEMPLATE_V2.exists():
    legacy = read_jsonl(LEGACY_TEMPLATE_V2)
    print(f"Legacy template-v2 unique IDs: {len({row['id'] for row in legacy})}")

## 7. Enforce the human-review gate

A review record is accepted only when it has a reviewer and all three verification flags are true. Do not set these fields automatically. If any sampled case fails review, correct or replace it before building the dataset.

In [ ]:
reviews = read_jsonl(REVIEW_LOG)

def review_accepted(row):
    return bool(
        row.get("reviewer")
        and row.get("scores_verified")
        and row.get("feedback_verified")
        and row.get("historical_accuracy_verified")
    )

accepted_reviews = [row for row in reviews if review_accepted(row)]
pending_reviews = [row for row in reviews if not review_accepted(row)]
print(f"Review queue: {len(reviews)} total, {len(accepted_reviews)} accepted, {len(pending_reviews)} pending/failed")
if pending_reviews:
    print("STOP: complete artifacts/reviews/synthetic_v2.jsonl locally, commit the reviewed log, then rerun.")
    print("Pending case IDs:", [row.get("case_id") for row in pending_reviews])

## 8. Apply reviews and build the immutable v2 mix

With 87 realistic rows and a 15% adversarial share, the largest current mix is 102 rows. This is a pilot retrain, not a 200-row checkpoint.

In [ ]:
assert reviews, f"Missing review log: {REVIEW_LOG}"
assert not pending_reviews, "Human-review gate is incomplete; refusing to build training data."

subprocess.run([
    sys.executable, "scripts/review_synthetic_v2.py",
    "--input", str(UNREVIEWED),
    "--review-log", str(REVIEW_LOG),
    "--output", str(REVIEWED),
], check=True)

realistic_rows = row_count(REVIEWED)
adversarial_rows = sum(
    row.get("failure_type") in {"grade_inflation_request", "prompt_injection"}
    for row in read_jsonl(LEGACY_ADVERSARIAL)
)
possible = [
    count for count in range(1, TARGET_CAP + 1)
    if count - round(count * ADVERSARIAL_RATIO) <= realistic_rows
    and round(count * ADVERSARIAL_RATIO) <= adversarial_rows
]
assert possible, "No eligible v2 training mix can be assembled."
target_count = max(possible)
print(f"Building {target_count} rows from {realistic_rows} realistic and {adversarial_rows} eligible adversarial rows.")

command = [
    sys.executable, "scripts/build_v2_artifacts.py",
    "--realistic", str(REVIEWED),
    "--adversarial", str(LEGACY_ADVERSARIAL),
    "--output-dir", str(V2_DIR),
    "--target-count", str(target_count),
    "--adversarial-ratio", str(ADVERSARIAL_RATIO),
]
if MANIFEST.exists() and TRAIN_CHAT.exists() and not FORCE_REBUILD:
    print(f"Using existing immutable artifact: {MANIFEST}")
else:
    if FORCE_REBUILD:
        command.append("--force")
    subprocess.run(command, check=True)

## 9. Verify manifest hashes and release gates

In [ ]:
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
audit = manifest["audit"]
assert not audit.get("global_reasons"), audit.get("global_reasons")
assert not audit.get("rejected"), audit.get("rejected")
assert audit.get("human_review_rate", 0) >= 0.10

for record in manifest["artifacts"]:
    path = Path(record["path"])
    assert path.exists(), path
    payload = path.read_bytes()
    assert len(payload) == record["bytes"], path
    assert hashlib.sha256(payload).hexdigest() == record["sha256"], path
    if path.suffix == ".jsonl":
        assert row_count(path) == record["rows"], path

train_rows = row_count(TRAIN_CHAT)
assert train_rows > 0
assert TRAIN_CHAT.parent == V2_DIR, "Refusing the legacy root-level train_chat_v2.jsonl."
print(f"Manifest verified: {train_rows} canonical v2 SFT rows; human review rate={audit['human_review_rate']:.2%}")

## 10. Materialize available 200/500/1,200 checkpoints

Below 200 rows, the checkpoint plan is intentionally empty and the next cell runs only a clearly labeled pilot model.

In [ ]:
checkpoint_command = [
    sys.executable, "scripts/run_v2_checkpoints.py",
    "--data", str(TRAIN_CHAT),
    "--counts", *map(str, CHECKPOINT_COUNTS),
    "--model", BASE_MODEL,
    "--litmus", str(LITMUS),
    "--golden", str(GOLDEN),
    "--external", str(EXTERNAL),
    "--output-root", "artifacts/checkpoints/v2",
]
if RUN_CHECKPOINT_SWEEP:
    checkpoint_command.append("--execute")
subprocess.run(checkpoint_command, check=True)
plan = json.loads(Path("artifacts/checkpoints/v2/checkpoint_plan.json").read_text(encoding="utf-8"))
print(f"Materialized {len(plan.get('runs', []))} formal checkpoints from {train_rows} rows.")

## 11. Train the current v2 artifact

Steps are scaled to approximately three epochs with an effective batch size of 16, with a 30-step minimum for the small pilot.

In [ ]:
effective_batch = BATCH_SIZE * GRAD_ACCUM
max_steps = max(30, math.ceil(train_rows / effective_batch) * EPOCHS)
print(f"Training {train_rows} rows for {max_steps} steps; output={MODEL_DIR}")
subprocess.run([
    sys.executable, "scripts/train_qlora.py",
    "--model", BASE_MODEL,
    "--data", str(TRAIN_CHAT),
    "--output", str(MODEL_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--max-steps", str(max_steps),
    "--seed", "13",
], check=True)

## 12. Refresh deterministic litmus baselines and evaluate the actual base model

In [ ]:
subprocess.run([
    sys.executable, "-m", "apush_frq_grader_slm.cli.run_eval",
    "--eval-path", str(LITMUS),
    "--output-dir", str(EVAL_DIR),
], check=True)

if RUN_BASE_EVAL:
    subprocess.run([
        sys.executable, "scripts/eval_hf_model.py",
        "--model", BASE_MODEL,
        "--model-name", "qwen_base_v2_litmus",
        "--eval-path", str(LITMUS),
        "--output-dir", str(EVAL_DIR),
    ], check=True)

## 13. Evaluate tuned v2 tracks separately

Missing tracks are skipped. College Board golden evaluation runs only when a private permission record grants evaluation. The notebook never falls back to the contaminated legacy `eval_real_cases.jsonl`.

In [ ]:
def run_hf_eval(label, path, real_eval=False):
    if not path.exists():
        print(f"SKIP {label}: missing {path}")
        return
    command = [
        sys.executable, "scripts/eval_hf_model.py",
        "--model", str(MODEL_DIR),
        "--model-name", f"{MODEL_NAME}_{label}",
        "--eval-path", str(path),
        "--output-dir", str(EVAL_DIR),
    ]
    if real_eval:
        command.append("--real-eval")
    subprocess.run(command, check=True)

run_hf_eval("litmus", LITMUS)
run_hf_eval("challenge", CHALLENGE)
run_hf_eval("external", EXTERNAL, real_eval=True)

golden_allowed = False
if PERMISSION.exists():
    permission = json.loads(PERMISSION.read_text(encoding="utf-8"))
    golden_allowed = permission.get("status") == "granted" and "evaluation" in permission.get("allowed_uses", [])
if GOLDEN.exists() and golden_allowed:
    run_hf_eval("golden", GOLDEN, real_eval=True)
else:
    print("SKIP golden: verified corpus and written evaluation permission are both required.")

## 14. Aggregate summaries without exposing holdout responses

In [ ]:
summary_rows = []
for path in sorted(EVAL_DIR.glob("*_summary.jsonl")):
    for row in read_jsonl(path):
        summary_rows.append({"file": path.name, **row})

if not summary_rows:
    print("No evaluation summaries found.")
else:
    preferred = [
        "model_name", "count", "structured_output_valid_rate",
        "rubric_accuracy_mean", "total_within_one_rate",
        "total_score_mae", "quadratic_weighted_kappa",
        "evidence_grounding_rate", "no_hallucination_rate", "robustness_mean",
    ]
    for row in summary_rows:
        print("\n", row["file"])
        print({key: row.get(key) for key in preferred if key in row})

report = {
    "repository_commit": commit,
    "base_model": BASE_MODEL,
    "model_path": str(MODEL_DIR),
    "manifest_path": str(MANIFEST),
    "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    "training_rows": train_rows,
    "max_steps": max_steps,
    "golden_evaluation_allowed": golden_allowed,
    "summaries": summary_rows,
}
report_path = EVAL_DIR / "v2_run_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Run report:", report_path)

## 15. Optional: save adapter, manifest, and aggregate report to Drive

Do not copy private permission records or official per-case outputs into shared storage.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/slm_models/apush-frq-grader-v2")
backup.mkdir(parents=True, exist_ok=True)
subprocess.run(["cp", "-r", str(MODEL_DIR), str(backup / "adapter")], check=True)
subprocess.run(["cp", str(MANIFEST), str(backup / MANIFEST.name)], check=True)
subprocess.run(["cp", str(EVAL_DIR / "v2_run_report.json"), str(backup / "v2_run_report.json")], check=True)
print("Saved:", backup)